In [1]:
# setup cv
import os
os.environ["OPENCV_VIDEOIO_MSMF_ENABLE_HW_TRANSFORMS"] = "0"

# policy
from lerobot.configs.policies import PreTrainedConfig

# robots
from lerobot.cameras.opencv.configuration_opencv import OpenCVCameraConfig
from lerobot.robots.so101_follower import SO101FollowerConfig

# record utils
from lerobot.record import record, RecordConfig, DatasetRecordConfig

# torch
from torch import cuda

# utils
from dotenv import load_dotenv

# my code
from src.paths import CALIBS_DIR, REPO_ROOT, HF_NAME, POLICIES_DIR, EVAL_DIR
from src.const import TABLE_START_POSE_OPEN

# set up env secrets
load_dotenv(REPO_ROOT / ".env", override=True)

# cuda
device = "cuda" if cuda.is_available() else "cpu"
print(f"Using device: {device}")

%load_ext autoreload
%autoreload 2

C:\Users\jonathan\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


Set Params:

In [2]:
PUSH_TO_HUB     = False
SAVE_TO_DATASET = True
REPO_NAME         = 'so101_leg_pick_and_place'
EXPERIMENT_NAME   = '50_episodes_v2'
POLICY_CHECKPOINT = '100000'
POLICY_TYPE = 'act'
NUM_EPISODES = 5
FPS = 30
EPISODE_TIME_SEC = 6000
RESET_TIME_SEC = 10
EVAL_TYPE = 'test'

Log to HF if pushing:

In [3]:
if PUSH_TO_HUB:
    raise NotImplementedError

Initialize Policy:

In [4]:
policy_path = POLICIES_DIR / POLICY_TYPE / REPO_NAME / EXPERIMENT_NAME / 'checkpoints' / POLICY_CHECKPOINT / 'pretrained_model'
policy_config = PreTrainedConfig.from_pretrained(policy_path)
policy_config.pretrained_path = policy_path

Robots Config

In [5]:
camera_config = {
    "wrist_cam": OpenCVCameraConfig(index_or_path=0, width=640, height=480, fps=30),
    "top_cam": OpenCVCameraConfig(index_or_path=1, width=640, height=480, fps=30),
}

robot_config = SO101FollowerConfig(
    port="COM7",
    id="so_101_follower",
    cameras = camera_config,
    calibration_dir = CALIBS_DIR
)

Build Dataset:

In [6]:
dataset_path = EVAL_DIR / POLICY_TYPE / REPO_NAME / f"{EXPERIMENT_NAME}-{EVAL_TYPE}"

resume = True if dataset_path.exists() else False
if resume and not any(dataset_path.iterdir()):
    dataset_path.rmdir()
    resume = False

dscfg = DatasetRecordConfig(
    repo_id                             = f"{HF_NAME}/eval_{REPO_NAME}_{EXPERIMENT_NAME}_{EVAL_TYPE}",
    single_task                         = f"eval dataset for {REPO_NAME} with policy {EXPERIMENT_NAME}, mode = {EVAL_TYPE}",
    root                                = dataset_path.__str__(),
    fps                                 = FPS,
    episode_time_s                      = EPISODE_TIME_SEC,
    reset_time_s                        = RESET_TIME_SEC,
    num_episodes                        = NUM_EPISODES,
    video                               = True,
    push_to_hub                         = PUSH_TO_HUB,
    private                             = True,
    num_image_writer_processes          = 2,
    num_image_writer_threads_per_camera = 4,
    video_encoding_batch_size           = 1,
)

rc = RecordConfig(
    robot        = robot_config,
    dataset      = dscfg,
    teleop       = None,
    policy       = policy_config,
    display_data = True,
    play_sounds  = True,
    resume       = resume
)

In [7]:
record(rc, reset_pose = TABLE_START_POSE_OPEN, give_feedback = False)

WARNING 2025-09-24 11:16:12 deo_utils.py:40 'torchcodec' is not available in your platform, falling back to 'pyav' as a default decoder


Loading weights from local directory


INFO 2025-09-24 11:16:15 a_opencv.py:181 OpenCVCamera(0) connected.
INFO 2025-09-24 11:16:16 a_opencv.py:181 OpenCVCamera(1) connected.
INFO 2025-09-24 11:16:16 follower.py:104 so_101_follower SO101Follower connected.
INFO 2025-09-24 11:16:17 ls\utils.py:227 Recording episode 2
INFO 2025-09-24 11:17:44 ls\utils.py:227 Reset the environment


Right arrow key pressed. Exiting loop...


Map: 100%|██████████| 1949/1949 [00:00<00:00, 2458.30 examples/s]
Generating train split: 8977 examples [00:00, 533106.80 examples/s]
Generating train split: 3 examples [00:00, 239.45 examples/s]
INFO 2025-09-24 11:18:36 ls\utils.py:227 Recording episode 3
INFO 2025-09-24 11:19:46 ls\utils.py:227 Reset the environment


Left arrow key pressed. Exiting loop and rerecord the last episode...


INFO 2025-09-24 11:19:48 ls\utils.py:227 Re-record episode
INFO 2025-09-24 11:19:50 ls\utils.py:227 Recording episode 3


Escape key pressed. Stopping data recording...


Map: 100%|██████████| 1923/1923 [00:00<00:00, 2063.48 examples/s]
Generating train split: 10900 examples [00:00, 816493.38 examples/s]
Generating train split: 4 examples [00:00, 216.78 examples/s]
INFO 2025-09-24 11:23:00 ls\utils.py:227 Stop recording
INFO 2025-09-24 11:23:05 a_opencv.py:487 OpenCVCamera(0) disconnected.
INFO 2025-09-24 11:23:05 a_opencv.py:487 OpenCVCamera(1) disconnected.
INFO 2025-09-24 11:23:05 follower.py:230 so_101_follower SO101Follower disconnected.
INFO 2025-09-24 11:23:05 ls\utils.py:227 Exiting


LeRobotDataset({
    Repository ID: 'jonathm126/eval_so101_leg_pick_and_place_50_episodes_v2_test',
    Number of selected episodes: '4',
    Number of selected samples: '10900',
    Features: '['action', 'observation.state', 'observation.images.wrist_cam', 'observation.images.top_cam', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index']',
})',